# ArguScore v3 — swap-invariant training on real debate outcomes

Replaces the v2 adapter, trained on IBM argument-quality pairs whose labels came
from a keyword heuristic. Each v2 defect below maps to a measured failure of the
deployed model.

| v2 defect | Measured consequence | Fix here |
|---|---|---|
| Labels from `compute_scores()` — keyword bonuses on `wa*10`, plus `±0.4` random noise against a 0.5 decision margin | 54.4% of scores in the dead 7.0–8.5 band | Target is the human-voted winner. No synthetic score, no noise. |
| `relevance_base` fed `length_factor` in directly | Margin tracked length (ρ=0.402) above human judgement (ρ=0.321) | Winner decorrelated from length inside every length quartile |
| Standalone arguments, one label order | Verdict flipped on **81.2%** of debates when labels were exchanged | Every debate trained twice, labels exchanged, target flipped |
| `max_seq_length=2048` | The median real debate was truncated | 4096 (97% of rows fit at 8192 if you prefer) |

**What this tests:** counterbalancing moved from inference to training. v2 needs
two passes to cancel its label preference. If this works, v3 needs one.

**Checkpoints upload to the Hub every `SAVE_STEPS`**, so a session that dies is
resumed by setting `RESUME_STEP` and re-running. Same pattern as v2.

**Before running:** upload `train.jsonl`, `val.jsonl`, `test.jsonl` as a Kaggle
dataset named `argus-ddo-train` (build with `python -m eval.build_training_set`).
The 140 debates reported on in the paper are excluded — verified, zero overlap.

In [ ]:
%%bash
pip install unsloth trl==0.15.2 peft datasets huggingface_hub -q
echo done

In [ ]:
# ── 1. Config, Hub, resume ───────────────────────────────────────────────────
import os, json, re, math, time, shutil, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
api = HfApi()

HF_REPO        = "s7nket/ArguScore-4B"
CHECKPOINT_DIR = "/kaggle/working/checkpoints-v3"
OUT_DIR        = "/kaggle/working/v3"
LOG_FILE       = "/kaggle/working/argus_v3_log.txt"
# Located rather than hard-coded: Kaggle slugifies a dataset title (capitals and
# spaces become dashes) and sometimes nests the files a level deeper, so the
# literal path is wrong as often as it is right. Find train.jsonl and use its
# directory.
import glob
_hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
DATA_DIR = os.path.dirname(_hits[0]) if _hits else "/kaggle/input/argus-ddo-train"
if _hits:
    print("found dataset at", DATA_DIR)
else:
    print("no train.jsonl anywhere under /kaggle/input — is the dataset attached?")
    print("contents:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "no /kaggle/input")

# Same 4B base as v2, which keeps the ArguScore-4B name consistent across the
# paper, the Hub repo and the serving notebook, and is known to load in this
# environment.
#
# It is reasoning-tuned, which on v2 meant the model narrated before answering
# and needed a <think></think> prefill to suppress. That concern largely goes
# away here: 6,608 examples whose target is a single line of JSON train the
# habit out, rather than fighting it at inference. If the eval in cell 6 still
# shows unparseable output, that is the first thing to revisit.
#
# 4B at 8192 also sits comfortably inside a 15 GB T4, where 8B at the same
# context runs at the memory edge.
MODEL_PATH     = "unsloth/Llama-3.1-Nemotron-Nano-4B-v1.1-bnb-4bit"

MAX_SEQ_LENGTH = 8192      # holds 97.4% of debates whole; 4096 holds 68%
LORA_RANK      = 32        # 16 was fitting a keyword function; reading a debate is harder
LORA_ALPHA     = 32
LEARNING_RATE  = 1e-4
NUM_EPOCHS     = 3
BATCH, ACCUM   = 1, 8      # effective batch 8; long contexts leave no room for more
SAVE_STEPS     = 150       # checkpoint + Hub upload cadence

RESUME_STEP    = None      # set to a step number to continue a killed session
RESUME_FROM    = None

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
try:
    api.create_repo(repo_id=HF_REPO, repo_type="model", private=True, exist_ok=True)
except Exception as e:
    print("repo note:", e)

if RESUME_STEP is not None:
    from huggingface_hub import snapshot_download
    local = snapshot_download(repo_id=HF_REPO, repo_type="model", token=HF_TOKEN,
                              allow_patterns=[f"v3/checkpoint-{RESUME_STEP}/*"])
    src = f"{local}/v3/checkpoint-{RESUME_STEP}"
    if os.path.exists(src):
        RESUME_FROM = f"{CHECKPOINT_DIR}/checkpoint-{RESUME_STEP}"
        shutil.copytree(src, RESUME_FROM, dirs_exist_ok=True)
        print(f"resuming from step {RESUME_STEP}")
    else:
        raise FileNotFoundError(f"no checkpoint-{RESUME_STEP} in {HF_REPO}/v3")

print(f"{MODEL_PATH}\n  seq={MAX_SEQ_LENGTH} rank={LORA_RANK} lr={LEARNING_RATE} "
      f"epochs={NUM_EPOCHS} | resume={RESUME_STEP or 'fresh'}")
for f in ("train", "val", "test"):
    p = f"{DATA_DIR}/{f}.jsonl"
    print(f"  {f}.jsonl: {'found' if os.path.exists(p) else 'MISSING — attach the dataset'}")

In [ ]:
# ── 2. Data, and proof the swap pairs survived ───────────────────────────────
from datasets import load_dataset
from collections import Counter, defaultdict

ds = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "val":   f"{DATA_DIR}/val.jsonl",
})

# The paired rows are the whole contribution, so verify rather than assume: each
# debate must appear exactly twice with opposite targets. If augmentation broke
# in transit this silently trains an ordinary biased model and nothing
# downstream would say so.
def _w(row):
    return json.loads(row["messages"][-1]["content"])["winner"]

pairs = defaultdict(list)
for r in ds["train"]:
    pairs[r["id"]].append(r)
bad = [k for k, v in pairs.items() if len(v) != 2 or _w(v[0]) == _w(v[1])]
print(f"train rows {len(ds['train'])}, debates {len(pairs)}, malformed pairs {len(bad)}")
assert not bad, f"{len(bad)} debates are not a proper swapped pair"
print("target balance:", dict(Counter(_w(r) for r in ds["train"])), "— even by construction")
print("\nexample prompt:\n", ds["train"][0]["messages"][1]["content"][:260], "...")
print("target:", ds["train"][0]["messages"][-1]["content"])

In [ ]:
# ── 3. Model ─────────────────────────────────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,                       # Unsloth's fused path is exact at 0
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",   # needed to fit this context on a T4
    random_state=17,
)

# v2 shipped with a tokenizer whose stop token resolved to None, so generation
# never halted and ran the full budget on every call. Resolve it from the
# tokenizer rather than hard-coding a Llama token, so swapping the base model
# (Qwen and Gemma use entirely different stop tokens) does not silently
# reintroduce that bug.
EOT = tokenizer.eos_token_id
if isinstance(EOT, (list, tuple)):
    EOT = EOT[0]
if EOT is None:
    for cand in ("<|eot_id|>", "<|im_end|>", "<end_of_turn>", "</s>"):
        tid = tokenizer.convert_tokens_to_ids(cand)
        if tid is not None and tid >= 0:
            EOT = tid
            break
assert EOT is not None, "no usable eos token — generation would never halt"
print(f"eos = {EOT} ({tokenizer.convert_ids_to_tokens(EOT)})")

In [ ]:
# ── 4. Train, checkpointing to the Hub ───────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False)
                     for m in batch["messages"]]}

train_ds = ds["train"].map(to_text, batched=True, remove_columns=ds["train"].column_names)
val_ds   = ds["val"].map(to_text,   batched=True, remove_columns=ds["val"].column_names)

steps = math.ceil(len(train_ds) / (BATCH * ACCUM)) * NUM_EPOCHS
print(f"{len(train_ds)} rows -> {steps} optimiser steps "
      f"(~{steps*2/3600:.1f}-{steps*4/3600:.1f} h)")
print(f"checkpoint every {SAVE_STEPS} steps -> resume by setting RESUME_STEP")


class HubCheckpoint(TrainerCallback):
    """Push each checkpoint to the Hub and print the step to resume from.

    Kaggle kills a session at 12 hours with no warning, and a LoRA checkpoint
    left in /kaggle/working dies with it. Uploading on save is what makes a long
    run survivable across sessions.
    """

    def __init__(self, path):
        self.path, self.start = path, time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        total = state.max_steps or steps
        done = max(state.global_step, 1)
        elapsed = (time.time() - self.start) / 60
        eta = elapsed / done * (total - done)
        msg = (f"step {state.global_step:>5}/{total} | loss {logs.get('loss','-')} "
               f"| eval {logs.get('eval_loss','-')} | {elapsed:.0f}m elapsed, ETA {eta:.0f}m")
        print(msg)
        with open(self.path, "a") as fh:
            fh.write(msg + "\n")

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        src = f"{CHECKPOINT_DIR}/checkpoint-{step}"
        if not os.path.exists(src):
            return
        try:
            api.upload_folder(folder_path=src, path_in_repo=f"v3/checkpoint-{step}",
                              repo_id=HF_REPO, repo_type="model", token=HF_TOKEN,
                              commit_message=f"v3 checkpoint {step}")
            print(f"  uploaded checkpoint-{step} — RESUME_STEP = {step}")
        except Exception as e:
            print(f"  upload failed at step {step}: {e}")


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    callbacks=[HubCheckpoint(LOG_FILE)],
    args=TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),   # T4 is Turing: fp16 only
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,                        # Kaggle disk is 20 GB
        seed=17,
        report_to="none",
    ),
)
stats = trainer.train(resume_from_checkpoint=RESUME_FROM)
print(stats)

In [ ]:
# ── 6. Save and publish — BEFORE the eval ───────────────────────────────────
#
# The adapter is the one thing this session cannot regenerate. The eval that
# follows makes 500 generation calls and takes hours, so running it first
# risks the session being killed with the weights still only in GPU memory.
# Publish, then measure.
final = f"{OUT_DIR}/adapter-final"
model.save_pretrained(final)
tokenizer.save_pretrained(final)
api.upload_folder(folder_path=final, path_in_repo="v3/adapter-final",
                  repo_id=HF_REPO, repo_type="model", token=HF_TOKEN,
                  commit_message="v3 swap-invariant adapter")
print("published to", f"{HF_REPO}/v3/adapter-final")

print("""
Serving changes required before this adapter goes live:

  1. ADAPTER_PATH = f"{local}/v3/adapter-final"
  2. JUDGE_SYSTEM must match the training schema — v3 emits
       {"winner":..., "pro_total":..., "con_total":...}
     and NOT evidence/logic/relevance, because those were never supervised.
     Inventing three numbers from one signal is what gave v2 three criteria
     that were the same number in three hats.
  3. judge_agent must override the two totals rather than the three criteria.
""")

In [ ]:
# ── 7. Evaluate: accuracy AND flip rate, on held-out debates ────────────────
#
# Safe to re-run in a fresh session by loading the published adapter — nothing
# here depends on the training state still being live.
#
# Accuracy alone cannot tell you whether this worked. v2 scored 62.3% under one
# labelling and 44.9% under the other, so a single number describes the
# labelling rather than the model. Every test debate is judged twice.
FastLanguageModel.for_inference(model)

def flip_labels(txt):
    """Exchange the two speaker labels and nothing else."""
    return re.sub(r"^(PRO|CON):",
                  lambda m: "CON:" if m.group(1) == "PRO" else "PRO:",
                  txt, flags=re.M)

def judge(system, user):
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors="pt", add_special_tokens=False,
                    truncation=True, max_length=MAX_SEQ_LENGTH - 64).to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=64, do_sample=False,
                             eos_token_id=EOT, pad_token_id=EOT)
    body = tokenizer.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)
    m = re.search(r"\{.*?\}", body, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

test = [json.loads(l) for l in open(f"{DATA_DIR}/test.jsonl", encoding="utf-8")]
correct = flips = usable = failed = 0
for i, row in enumerate(test):
    system, user = row["messages"][0]["content"], row["messages"][1]["content"]
    truth = json.loads(row["messages"][-1]["content"])["winner"]
    a, b = judge(system, user), judge(system, flip_labels(user))
    if not a or not b:
        failed += 1
        continue
    usable += 1
    correct += (a.get("winner") == truth)
    # b saw exchanged labels, so map its verdict back: agreement means the same
    # real side won both times.
    back = {"pro": "con", "con": "pro"}.get(b.get("winner"), b.get("winner"))
    flips += (a.get("winner") != back)
    if (i + 1) % 25 == 0:
        print(f"  [{i+1}/{len(test)}] acc {correct/max(1,usable):.3f}  flip {flips/max(1,usable):.3f}")

print("\n" + "=" * 58)
print(f"v3 on {usable} held-out debates ({failed} unparseable)")
print(f"  single-pass accuracy   {correct/usable:.3f}")
print(f"  verdict flip rate      {flips/usable:.3f}")
print(f"\n  v2 baseline:           accuracy 0.623, flip rate 0.812")
print(f"  chance:                0.500")
print("  success = flip rate well under 0.812 with accuracy at or above 0.623")
print("=" * 58)